# Day 2 — end-to-end smoke test (generate → train → eval)

Proves the full loop runs on 50 junk examples (spec **S2.11**). Uses **Faker pattern-type negatives**
as the junk training data, so it needs **no teacher API key**. GPU runtime required.

The numbers here are meaningless (junk data, 1 epoch) — the point is that
generate → QLoRA train → base-vs-tuned eval runs end to end without errors. The real dataset and
training land on Day 3.

In [ ]:
import os, sys

if not os.path.isdir("src/common"):
    os.system("git clone https://github.com/f15cubing/slm-deid.git")
    if os.path.isdir("slm-deid"):
        os.chdir("slm-deid")
sys.path.insert(0, os.getcwd())
print("cwd:", os.getcwd())

!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" 2>/dev/null || pip install -q unsloth
!pip install -q trl datasets faker pyyaml

## 1. Generate 50 junk training examples (no API)

In [ ]:
from src.datagen.negatives import generate_negatives
from src.common.schema import write_jsonl

junk = generate_negatives(n=50, seed=0)
write_jsonl("data/splits/train.jsonl", junk)
write_jsonl("data/splits/val.jsonl", junk[:5])
print("wrote", len(junk), "junk training examples -> data/splits/train.jsonl")

## 2. QLoRA train (smoke: 1 epoch, ≤50 examples)

In [ ]:
from src.train.qlora import load_config, train

cfg = load_config("configs/train.yaml")
adapter = train(cfg, smoke=True)   # writes outputs/sft-v1-smoke
print("adapter:", adapter)

## 3. Base-vs-tuned eval on the quarantined hard cases

Loop closes here: same eval harness scores both models. (Smoke numbers are noise; Day 3 does the
real run.)

In [ ]:
from src.eval.run import evaluate, compare, _load_examples
from src.infer import load_hf_tagger

examples = _load_examples("eval/hardcases")
print(f"{len(examples)} quarantined hard-cases examples")

base = load_hf_tagger()
rep_base = evaluate(base, examples, label="base")

tuned = load_hf_tagger(adapter=adapter)
rep_tuned = evaluate(tuned, examples, label="tuned-smoke")

print("\n" + compare([rep_base, rep_tuned]))
print("\n[OK] S2.11 — generate -> train -> eval ran end to end.")